In [1]:
%cd /home/brimmann/works/xRAG

/home/brimmann/works/xRAG


/home/brimmann/works/xRAG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import torch.nn as nn
import re
from types import SimpleNamespace
import torch
from src.distill.models.projector import Projector

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
teacher_projector_config = SimpleNamespace(
    projector_type='mlp2x_gelu',
    retriever_hidden_size=4096,
    hidden_size=4096
)

teacher_projector = Projector(teacher_projector_config)
teacher_projector.load_state_dict(torch.load("tensorstorage/teacher_projector_weights.pth"))
teacher_projector.to(device)

Projector(
  (projector): Sequential(
    (0): Linear(in_features=4096, out_features=4096, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=4096, out_features=4096, bias=True)
  )
)

In [5]:
student_projector_config = SimpleNamespace(
    projector_type='mlp2x_gelu',
    retriever_hidden_size=4096,
    hidden_size=2304
)

student_projector = Projector(student_projector_config)
student_projector.to(device)

Projector(
  (projector): Sequential(
    (0): Linear(in_features=4096, out_features=2304, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=2304, out_features=2304, bias=True)
  )
)

In [6]:
projection_layer = nn.Linear(2304, 4096)
projection_layer.to(device)

Linear(in_features=2304, out_features=4096, bias=True)

In [7]:
import pickle
doc_embeds_list = None
with open('tensorstorage/all_doc_embeds.pkl', 'rb') as f:
    doc_embeds_list = pickle.load(f)

In [8]:
doc_embeds_tensors = [tensor[0] for tensor in doc_embeds_list]

In [9]:
stacked_embeds = torch.stack(doc_embeds_tensors)
stacked_embeds.shape

torch.Size([11313, 4096])

In [10]:
from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(stacked_embeds)

In [11]:
from torch.utils.data import random_split


train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [ ]:
import torch.optim as optim
import itertools
import os

# Set models to appropriate modes
student_projector.train()
projection_layer.train()
teacher_projector.eval() # Teacher model is not being trained

# We want to optimize the parameters of the student projector and the projection layer
params_to_optimize = itertools.chain(student_projector.parameters(), projection_layer.parameters())

# Initialize the optimizer
optimizer = optim.Adam(params_to_optimize, lr=1e-4) # You can tune the learning rate

num_epochs = 100
loss_function = torch.nn.MSELoss()
best_val_loss = float('inf')
output_dir = "tensorstorage/distilled_model"
os.makedirs(output_dir, exist_ok=True)

In [13]:

for epoch in range(num_epochs):
    total_train_loss = 0

    for batch in train_dataloader:
        x = batch[0].to(device)
        student_output = student_projector(x.to(next(teacher_projector.parameters()).dtype))
        student_output_projected = projection_layer(student_output)
        teacher_output = None
        with torch.no_grad():
            teacher_output = teacher_projector(x.to(next(teacher_projector.parameters()).dtype))

        loss = loss_function(student_output_projected, teacher_output.detach())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)

    student_projector.eval()
    projection_layer.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_dataloader:
            x = batch[0].to(device)
            student_output = student_projector(x.to(next(teacher_projector.parameters()).dtype))
            student_output_projected = projection_layer(student_output)
            teacher_output = teacher_projector(x.to(next(teacher_projector.parameters()).dtype))

            val_loss = loss_function(student_output_projected, teacher_output)
            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(val_dataloader)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')


    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"New best model found at epoch {epoch+1} with validation loss: {best_val_loss:.4f}. Saving model.")
        torch.save(student_projector.state_dict(), os.path.join(output_dir, "best_student_projector.pth"))
        torch.save(projection_layer.state_dict(), os.path.join(output_dir, "best_projection_layer.pth"))





New best model found at epoch 1 with validation loss: 14666.5993. Saving model.
New best model found at epoch 2 with validation loss: 12853.5933. Saving model.
New best model found at epoch 3 with validation loss: 11828.2849. Saving model.
New best model found at epoch 4 with validation loss: 11025.3807. Saving model.
New best model found at epoch 5 with validation loss: 10484.0423. Saving model.
New best model found at epoch 6 with validation loss: 10060.1481. Saving model.
New best model found at epoch 7 with validation loss: 9651.5323. Saving model.
New best model found at epoch 8 with validation loss: 9215.0391. Saving model.
New best model found at epoch 9 with validation loss: 8811.6446. Saving model.
Epoch [10/10], Train Loss: 8395.4080, Val Loss: 8404.4380
New best model found at epoch 10 with validation loss: 8404.4380. Saving model.
